# Speaker Inversion — Colab end-to-end (clone → convert → train)

Trains a **Speaker Inversion** embedding for one character of the
[`l2533584225/Inversion-speaker-dataset`](https://huggingface.co/datasets/l2533584225/Inversion-speaker-dataset)
against the frozen [`Aratako/Irodori-TTS-v4.1-Small`](https://huggingface.co/Aratako/Irodori-TTS-v4.1-Small)
base model. The base model never changes; only a `(16, 768)` token table is learned.

What this notebook does, in order:

1. installs the system deps (ffmpeg) and syncs the Python env with `uv` (`--extra cu128`, T4-friendly)
2. clones [Irodori-TTS-trainer](https://github.com/lijunjie2232/Irodori-TTS-trainer)
3. downloads the dataset and the base checkpoint from HuggingFace — both are gitignored in the repo
4. converts the VAD/ASR segments into DACVAE latents + a training manifest, written to
   `<dataset>/<character>/converted/`
5. optionally pushes **only** that `converted/` directory back to the HF dataset repo
6. runs Speaker Inversion training — single process on 1 GPU, `torchrun` DDP on 2

Every knob lives in the **CONFIG** cell below: paths, HF repos, convert args, train args.
Nothing else needs editing.

Runtime: **GPU (T4)**. Set `UPDATE_DATASET = True` only if you have an `HF_TOKEN` Colab secret
with write access to the dataset repo.

> Mirrors `speaker-inversion-dataset/vad_asr.ipynb`, which produces the `results/<character>/*.jsonl`
> segments this notebook consumes.

## 0. Config

In [ ]:
# ===========================================================================
# CONFIG — every knob this notebook uses. Nothing below needs editing.
# ===========================================================================
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

# ---- repo ----------------------------------------------------------------
REPO_URL    = "https://github.com/lijunjie2232/Irodori-TTS-trainer"
REPO_BRANCH = "main"
REPO_DIR    = Path("/content/Irodori-TTS-trainer")
UV_EXTRA    = "cu128"        # T4 is sm_75, covered by the cu128 wheels

# ---- character -----------------------------------------------------------
CHARACTER = "koyori"         # subdirectory under the dataset root

# ---- dataset (HuggingFace) ----------------------------------------------
HF_DATASET_REPO = "l2533584225/Inversion-speaker-dataset"
HF_TOKEN_SECRET = "HF_TOKEN"    # Colab secret name; only read when UPDATE_DATASET
UPDATE_DATASET  = False         # True -> upload <dataset>/<character>/converted to HF
# <character>/* = source audio, results/<character>/* = the VAD+ASR segments
DATASET_INCLUDE = [f"{CHARACTER}/*", f"results/{CHARACTER}/*"]

# ---- models --------------------------------------------------------------
BASE_MODEL_REPO = "Aratako/Irodori-TTS-v4.1-Small"
CODEC_REPO      = "Aratako/Semantic-DACVAE-Japanese-32dim"   # or a local weights.pth path

# ---- convert args (speaker_inversion_example/scripts/convert_dataset.py) --
CONVERT_AUDIO_FILTER = ""       # substring of the source mp3 name; "" = every file
CONVERT_LIMIT        = 0        # 0 = all segments; >0 = first N written latents (smoke test)
CONVERT_MIN_SECONDS  = 1.0
CONVERT_MAX_SECONDS  = 30.0     # keep <= max_latent_steps / LATENT_FPS of the training config
CONVERT_NORMALIZE_DB = -16.0
CONVERT_DEVICE       = "cuda"
CONVERT_OVERWRITE    = False    # required when converted/latents already holds data

# ---- train args (train.py; these override configs/speaker_inversion.yaml) --
TRAIN_BATCH_SIZE  = 16          # PER PROCESS, so x2 on a 2-GPU runtime
TRAIN_GRAD_ACCUM  = 1
TRAIN_MAX_STEPS   = 3000
TRAIN_LR          = 0.01        # high on purpose: only 16x768 params are trainable
TRAIN_SAVE_EVERY  = 250
TRAIN_LOG_EVERY   = 20
TRAIN_NUM_WORKERS = 2
TRAIN_PRECISION   = "fp16"      # T4 has no bf16 but fp16 -> train.py falls back to fp32 on its own if bf16
TRAIN_NPROC       = 1           # 0 = auto (1 process per visible GPU), 1 = force single process (gpu)

# ---- derived (do not edit) ----------------------------------------------
EXAMPLE     = REPO_DIR / "speaker_inversion_example"
DATASET     = EXAMPLE / "data" / "speaker-inversion-dataset"
CONVERT     = DATASET / CHARACTER / "converted"
MANIFEST    = CONVERT / "manifest.jsonl"
LATENTS     = CONVERT / "latents"
REPORT      = CONVERT / "convert_report.json"
CONFIG_YAML = EXAMPLE / "configs" / "speaker_inversion.yaml"
BASE_CHECKPOINT = REPO_DIR / "models" / "Irodori-TTS-v4.1-Small" / "model.safetensors"
OUTPUT_DIR  = EXAMPLE / "outputs" / CHARACTER
LATENT_FPS  = 25.0              # DACVAE: 48 kHz / 1920-sample hop

RUN = ["uv", "run", "--no-sync"]   # runs inside the .venv created by `uv sync`


def run(*args, cwd=None):
    '''Echo and run a command. cwd falls back to the current dir before the clone exists.'''
    workdir = Path(cwd or REPO_DIR)
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(workdir if workdir.is_dir() else Path.cwd()), check=True)


for label, value in [("repo", REPO_DIR), ("dataset", DATASET), ("converted", CONVERT),
                     ("base ckpt", BASE_CHECKPOINT), ("outputs", OUTPUT_DIR)]:
    print(f"{label:<10} {value}")

## 1. System dependencies

`ffmpeg` / `ffprobe` are what `convert_dataset.py` uses to cut the segments out of the long
recordings. Colab already runs as root, so `apt-get` works without `sudo`. `uv` ships with Colab.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!uv --version && ffmpeg -version | head -1

## 2. Clone the trainer

`--recursive` is deliberately **not** used: `third_part/*` (CrispASR / FunASR) is only needed by
the VAD/ASR stage, not by Speaker Inversion training.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    run("git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH)
    run("git", "-C", REPO_DIR, "checkout", "FETCH_HEAD")
else:
    run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR)

## 3. Python environment

`uv sync` creates `<repo>/.venv` and resolves `torch` / `torchaudio` from the `pytorch-cu128`
index declared in the repo's own `pyproject.toml`. `dacvae` comes from git (`uv.sources`),
so it is not on PyPI — `uv` handles that too.

In [ ]:
run("uv", "sync", "--extra", UV_EXTRA, "--no-dev")
run(*RUN, "python", "-c",
    "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda,"
    " '| available', torch.cuda.is_available(), '|', torch.cuda.device_count(), 'GPU(s)')")

## 4. Data + base checkpoint

Both are gitignored in the trainer repo (root `.gitignore` ignores `speaker_inversion_example/data`
and `models/`), so they are fetched from HuggingFace separately. The `koyori/*.mp3` set alone is
~4 GB — narrow `DATASET_INCLUDE` if you only want a quick pass.

In [ ]:
DATASET.mkdir(parents=True, exist_ok=True)
include = [flag for pattern in DATASET_INCLUDE for flag in ("--include", pattern)]
run("uvx", "hf", "download", "--type=dataset", HF_DATASET_REPO, "--local-dir", DATASET, *include)

# 3 GB base checkpoint the embedding is trained against; ~/.cache/huggingface is reused on re-runs.
run("uvx", "hf", "download", BASE_MODEL_REPO,
    "--local-dir", REPO_DIR / "models" / "Irodori-TTS-v4.1-Small")

print("\ndataset root:", sorted(p.name for p in DATASET.iterdir()))

## 5. Convert segments → latents + manifest

`convert_dataset.py` resolves each mp3 by file name (the `audio_path` field in the jsonl carries a
stale `data/` prefix), cuts `[start_ms, end_ms)` with one ffmpeg seek per segment, and encodes it
with the same `DACVAECodec` as `prepare_manifest.py`. Output is field-for-field the manifest
`train.py` expects:

```
<dataset>/<character>/converted/
  manifest.jsonl          {"text", "latent_path": "latents/00000000.pt", "num_frames"}
  latents/00000000.pt     (T, 32) float32
  convert_report.json     written in section 6
```

In [ ]:
convert_config = [
    "--dataset-root", str(DATASET),
    "--speaker", CHARACTER,
    "--manifest", str(MANIFEST),
    "--latent-dir", str(LATENTS),
    "--codec-repo", CODEC_REPO,
    "--min-seconds", CONVERT_MIN_SECONDS,
    "--max-seconds", CONVERT_MAX_SECONDS,
    "--normalize-db", CONVERT_NORMALIZE_DB,
    "--device", CONVERT_DEVICE,
]
if CONVERT_AUDIO_FILTER:
    convert_config += ["--audio", CONVERT_AUDIO_FILTER]
if CONVERT_LIMIT:
    convert_config += ["--limit", CONVERT_LIMIT]
if CONVERT_OVERWRITE:
    convert_config += ["--overwrite"]
convert_config

In [ ]:
run(*RUN, "python", EXAMPLE / "scripts" / "convert_dataset.py", *convert_config)

## 6. Metadata + optional HuggingFace sync

`convert_report.json` records the conversion provenance so the `converted/` directory is
self-describing once it lands on the Hub. The assertions are the cheap sanity check that the
manifest and the latents agree.

`UPDATE_DATASET = True` uploads **only** `<character>/converted/` — the source audio and the
VAD/ASR results already live on the Hub and are not re-uploaded.

In [ ]:
records = [json.loads(line) for line in
           MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
referenced = [MANIFEST.parent / r["latent_path"] for r in records]

missing = [str(p) for p in referenced if not p.is_file()]
assert not missing, f"{len(missing)} latent(s) referenced by the manifest are missing, e.g. {missing[:3]}"
assert all(r["num_frames"] > 0 for r in records), "manifest contains an empty latent"
assert len({r["latent_path"] for r in records}) == len(records), "duplicate latent_path in manifest"

total_frames = sum(r["num_frames"] for r in records)


# CONVERT_AUDIO_FILTER narrows which source files were used, so mirror it here.
def used(p):
    return not CONVERT_AUDIO_FILTER or CONVERT_AUDIO_FILTER in p.name


report = {
    "character": CHARACTER,
    "dataset_repo": HF_DATASET_REPO,
    "source_audio": sorted(p.name for p in (DATASET / CHARACTER).glob("*.mp3") if used(p)),
    "source_segments": sorted(p.name for p in (DATASET / "results" / CHARACTER).glob("*.jsonl") if used(p)),
    "manifest": MANIFEST.name,
    "latent_dir": LATENTS.name,
    "n_samples": len(records),
    "total_frames": total_frames,
    "total_seconds": round(total_frames / LATENT_FPS, 2),
    "latent_bytes": sum(p.stat().st_size for p in referenced),
    "codec_repo": CODEC_REPO,
    "codec_sample_rate": 48000,
    "latent_dim": 32,
    "latent_fps": LATENT_FPS,
    "normalize_db": CONVERT_NORMALIZE_DB,
    "min_seconds": CONVERT_MIN_SECONDS,
    "max_seconds": CONVERT_MAX_SECONDS,
    "limit": CONVERT_LIMIT,
    "audio_filter": CONVERT_AUDIO_FILTER,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))

if UPDATE_DATASET:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get(HF_TOKEN_SECRET)
    run("uvx", "hf", "upload", HF_DATASET_REPO, CONVERT,
        f"{CHARACTER}/converted", "--repo-type=dataset")
else:
    print("\nUPDATE_DATASET is False — skipping the HuggingFace upload.")

## 7. Train

The base model is frozen; the run trains only a `(16, 768)` embedding — watch for this line in the
log:

```
Speaker Inversion freeze applied: trainable=12,288 frozen=766,052,385
```

Two things worth knowing:

- **DDP.** `torchrun` is used only when more than one GPU is visible; with one GPU the notebook
  runs plain `python train.py`. `batch_size` is per process, so 2 GPUs means the effective batch
  doubles.
- **Precision.** T4 (sm_75) has no bf16 support, so `train.py` prints a warning and falls back to
  fp32 by itself. Set `TRAIN_PRECISION = "fp32"` to skip the warning.

In [ ]:
train_config = [
    "--config", str(CONFIG_YAML),
    "--manifest", str(MANIFEST),
    "--init-checkpoint", str(BASE_CHECKPOINT),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--batch-size", TRAIN_BATCH_SIZE,
    "--gradient-accumulation-steps", TRAIN_GRAD_ACCUM,
    "--max-steps", TRAIN_MAX_STEPS,
    "--lr", TRAIN_LR,
    "--save-every", TRAIN_SAVE_EVERY,
    "--log-every", TRAIN_LOG_EVERY,
    "--num-workers", TRAIN_NUM_WORKERS,
    "--precision", TRAIN_PRECISION,
]

import torch

n_gpu = torch.cuda.device_count()
nproc = TRAIN_NPROC or n_gpu
if nproc > 1:
    print(f"DDP over {nproc} GPU(s); batch_size={TRAIN_BATCH_SIZE} is per process "
          f"-> global batch {TRAIN_BATCH_SIZE * nproc}")
    train_cmd = [*RUN, "torchrun", "--nproc_per_node", nproc, "train.py", *train_config]
else:
    print(f"single process ({n_gpu} GPU visible)")
    train_cmd = [*RUN, "python", "train.py", *train_config]

train_config

In [ ]:
run(*train_cmd)

## 8. Outputs

In [ ]:
final_embed = OUTPUT_DIR / "checkpoint_final.speaker.safetensors"
assert final_embed.is_file(), f"training produced no embedding at {final_embed}"

print(f"converted -> {CONVERT}")
!ls -alh {CONVERT}
print(f"\nembeddings -> {OUTPUT_DIR}")
!ls -alh {OUTPUT_DIR}

## Using the embedding

The embedding is only a few KB and only means anything together with the **same base checkpoint**
it was trained against. The `.speaker.safetensors` suffix is what tells the codebase this is an
embedding rather than a full model:

```bash
uv run --no-sync python infer.py \
  --checkpoint models/Irodori-TTS-v4.1-Small/model.safetensors \
  --ref-embed speaker_inversion_example/outputs/koyori/checkpoint_final.speaker.safetensors \
  --text "こんにちは、これは学習した話者埋め込みを使った推論です。" \
  --output-wav outputs/koyori.wav
```

`speaker_inversion_example/scripts/04_eval_snapshots.sh` renders the same sentence from every
periodic snapshot so you can keep whichever one sounds best.

**Not synced to the Hub:** only `converted/` is uploaded. If you want the trained embedding to
survive the Colab runtime, copy it out yourself (Drive, `hf upload`, …).